# Color pipeline validation

Validates the deterministic color conversion pipeline (`cnn_demosaic.color_convert`) against
the camera-processed sRGB color card references.

The training data files are stored with git LFS. Fetch them first:

```
git lfs pull --include "training/*"
```

The pipeline under test follows the dcraw chain:

1. Linearize the demosaiced image using the sensor black and white levels.
2. White balance with the as-shot camera multipliers (green normalized).
3. Convert camera RGB to linear sRGB using the camera color matrix from the raw file.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyexr
import rawpy

from cnn_demosaic import color_convert

# The path to the camera processed color card
SRGB_PATH = "../training/DSCF5652_card_srgb.exr"
# The path to the demosaiced but not color corrected color card
EXR_PATH = "../training/DSCF5652_card.exr"
# The path to the raw image
RAW_PATH = "../training/DSCF5652.RAF"


In [ ]:
with rawpy.imread(RAW_PATH) as raw_img:
    raw_img_arr = raw_img.raw_image.astype(np.float32).copy()
    norm_min = float(np.min(raw_img_arr))
    norm_max = float(np.max(raw_img_arr))
    color_params = color_convert.get_color_params(raw_img)

print("XYZ to camera matrix:")
print(color_params.xyz_to_cam)
print()
print("Camera to sRGB matrix:")
print(color_params.cam_to_srgb)
print()
print("As-shot WB multipliers (green normalized):", color_params.wb_multipliers)
print("Black level per channel:", color_params.black_level_per_channel)
print("White level:", color_params.white_level)

# The reciprocal of the un-normalized sRGB to camera composition row sums are
# the daylight white balance multipliers. These should closely match the
# camera reported daylight white balance.
srgb_to_cam = color_params.xyz_to_cam @ color_convert.srgb_to_xyz
pre_mul = 1.0 / srgb_to_cam.sum(axis=1)
print("WB multipliers implied by the color matrix:", pre_mul)
with rawpy.imread(RAW_PATH) as raw_img:
    print("Camera reported daylight WB:", raw_img.daylight_whitebalance)


## Convert the demosaiced color card

In [ ]:
card_arr = pyexr.read(EXR_PATH)[:, :, :3]

output_arr = color_convert.linearize(card_arr, color_params, norm_min, norm_max)
output_arr = color_convert.white_balance(output_arr, color_params.wb_multipliers)
output_arr = color_convert.camera_to_srgb(output_arr, color_params)
output_arr = np.clip(output_arr, 0.0, 1.0)


## Compare against the camera processed sRGB reference

In [ ]:
srgb_arr = pyexr.read(SRGB_PATH)[:, :, :3]

# The color card swatches are 160x160 tiles along the top of the image.
CHIP_SIZE = 160

def chip_means(img_arr, n_chips):
    return np.asarray([
        img_arr[:CHIP_SIZE, i * CHIP_SIZE:(i + 1) * CHIP_SIZE].reshape(-1, 3).mean(axis=0)
        for i in range(n_chips)
    ])

n_chips = min(srgb_arr.shape[1] // CHIP_SIZE, output_arr.shape[1] // CHIP_SIZE)
converted_means = chip_means(output_arr, n_chips)
reference_means = chip_means(srgb_arr, n_chips)

print("chip | converted (RGB) | reference (RGB)")
for i in range(n_chips):
    print(f"{i:4d} | {np.round(converted_means[i], 4)} | {np.round(reference_means[i], 4)}")

# Hues should match: compare the channel ratios of each chip.
ratios_converted = converted_means / converted_means[:, 1:2]
ratios_reference = reference_means / reference_means[:, 1:2]
print()
print("Max chip channel ratio error:", np.abs(ratios_converted - ratios_reference).max())


In [ ]:
def display_pair(img_a, img_b, title_a="converted", title_b="reference"):
    (_, axes) = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(np.clip(img_a, 0.0, 1.0), vmin=0, vmax=1)
    axes[0].set_title(title_a)
    axes[0].axis("off")
    axes[1].imshow(np.clip(img_b, 0.0, 1.0), vmin=0, vmax=1)
    axes[1].set_title(title_b)
    axes[1].axis("off")
    plt.show()

display_pair(output_arr, srgb_arr)


## Reference: dcraw processing via rawpy

For comparison, this is LibRaw's own rendering of the same raw file using the
as-shot white balance.

In [ ]:
with rawpy.imread(RAW_PATH) as raw_img:
    dcraw_rgb = raw_img.dcraw_process(rawpy.Params(use_camera_wb=True, output_bps=16))

dcraw_rgb = dcraw_rgb.astype(np.float32) / 65535.0
display_pair(output_arr, dcraw_rgb, title_a="converted", title_b="dcraw_process")
